In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Cross-engine verification — neutron powder, constant wavelength (Bragg)

This page calculates the **same** diffraction pattern for one structure
and one experiment with each supported engine, **without any fitting**,
and reports how closely the engines agree. It doubles as a regression
check run by `pixi run script-tests`.

In [2]:
import numpy as np

import easydiffraction as ed
from easydiffraction.analysis.calculators.support import calculator_support_matrix
from easydiffraction.analysis.fit_helpers.metrics import get_reliability_inputs

## Build the project (La0.5Ba0.5CoO3, HRPT)

In [3]:
project = ed.Project()
project.structures.add_from_cif_path(ed.download_data(id=1, destination='data'))
project.experiments.add_from_cif_path(ed.download_data(id=2, destination='data'))

experiment = project.experiments['hrpt']

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Getting data...


Data #1: La0.5Ba0.5CoO3 (crystal structure)


✅ Data #1 downloaded to '../../../data/ed-1.cif'


Getting data...


Data #2: La0.5Ba0.5CoO3, HRPT (PSI), 300 K


✅ Data #2 downloaded to '../../../data/ed-2.cif'


## Engines to compare

The calculator support matrix declares which engines can compute this
instrument condition (`cwl-pd`).

In [4]:
by_tag = {entry.instrument_tag: entry for entry in calculator_support_matrix()}
declared = sorted(c.value for c in by_tag['cwl-pd'].calculators)
print('Engines declared for cwl-pd:', declared)

ENGINES = ['cryspy', 'crysfml']

Engines declared for cwl-pd: ['crysfml', 'cryspy', 'pdffit']


## Calculate the pattern with each engine (no fitting)

In [5]:
y_calc_by_engine = {}
for engine in ENGINES:
    experiment.calculator.type = engine
    assert experiment.calculator.type == engine
    _, y_calc, _ = get_reliability_inputs(project.structures, [experiment])
    y_calc_by_engine[engine] = np.asarray(y_calc, dtype=float)
    # Per-engine measured-vs-calculated view (rendered in the docs build).
    project.display.pattern(expt_name='hrpt')

Calculator for experiment 'hrpt' already set to


cryspy


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Closeness metrics between engines

In [6]:
a = y_calc_by_engine['cryspy']
b = y_calc_by_engine['crysfml']

assert a.shape == b.shape, 'engines returned patterns of different length'
assert np.all(np.isfinite(a)), 'cryspy pattern has non-finite values'
assert np.all(np.isfinite(b)), 'crysfml pattern has non-finite values'

rms = float(np.sqrt(np.mean((a - b) ** 2)))
norm = float(np.sqrt(np.mean(a**2)))
profile_diff_pct = 100.0 * rms / norm if norm else float('nan')
max_deviation = float(np.max(np.abs(a - b)))
intensity_ratio = float(a.sum() / b.sum()) if b.sum() else float('nan')
correlation = float(np.corrcoef(a, b)[0, 1])

print(f'profile difference:                 {profile_diff_pct:.2f} %')
print(f'max point-wise deviation:           {max_deviation:.4g}')
print(f'integrated-intensity ratio (cp/cf): {intensity_ratio:.4f}')
print(f'Pearson correlation:                {correlation:.4f}')

profile difference:                 0.48 %
max point-wise deviation:           17.83
integrated-intensity ratio (cp/cf): 1.0017
Pearson correlation:                1.0000


## Overlay

Both engines on one chart, in distinct colours and line styles.

In [7]:
import plotly.graph_objects as go

x = np.arange(a.size)
fig = go.Figure()
fig.add_scatter(x=x, y=a, mode='lines', name='cryspy', line={'color': 'royalblue'})
fig.add_scatter(x=x, y=b, mode='lines', name='crysfml', line={'color': 'crimson', 'dash': 'dot'})
fig.update_layout(
    title='Calculated patterns: cryspy vs crysfml',
    xaxis_title='point index',
    yaxis_title='Icalc',
)
# Bare expression renders inline in the executed notebook; a no-op as a
# plain script, so `pixi run script-tests` stays headless-safe.
fig

## Regression assertions

Explicit, named tolerances for each metric. cryspy and crysfml agree
very closely here (a first measurement gives ≈0.5% profile difference,
intensity ratio ≈1.00, correlation ≈1.00), so these bounds keep a
generous cross-platform margin while still catching a real regression.
They are tightened further once multi-platform nightly runs establish
the spread for each engine pair.

In [8]:
MAX_PROFILE_DIFFERENCE_PCT = 10.0  # measured ≈0.5%
MAX_RELATIVE_DEVIATION = 1.0  # scale-sensitive; kept generous
MIN_INTENSITY_RATIO = 0.8  # measured ≈1.00
MAX_INTENSITY_RATIO = 1.25
MIN_CORRELATION = 0.99  # measured ≈1.00

peak = float(np.max(np.abs(a)))
relative_deviation = max_deviation / peak if peak else float('nan')

assert profile_diff_pct < MAX_PROFILE_DIFFERENCE_PCT, (
    f'profile difference {profile_diff_pct:.2f}% exceeds {MAX_PROFILE_DIFFERENCE_PCT}%'
)
assert relative_deviation < MAX_RELATIVE_DEVIATION, (
    f'relative max deviation {relative_deviation:.3f} exceeds {MAX_RELATIVE_DEVIATION}'
)
assert MIN_INTENSITY_RATIO < intensity_ratio < MAX_INTENSITY_RATIO, (
    f'integrated-intensity ratio {intensity_ratio:.3f} outside '
    f'[{MIN_INTENSITY_RATIO}, {MAX_INTENSITY_RATIO}]'
)
assert correlation > MIN_CORRELATION, (
    f'cross-engine correlation {correlation:.4f} below {MIN_CORRELATION}'
)